# DeiT-Tiny Final Evaluation — Google Colab

**CSCE5218 · Tharun / Dev / Jiong**

**Repository:** https://github.com/TharunSwaminathan/deit-sparse-attention  
**Reference PR:** https://github.com/TharunSwaminathan/deit-sparse-attention/pull/1  
**Recommended repository path:** `notebooks/final_evaluation.ipynb`

This notebook was developed from the November 26, 2026 repository snapshot. The current state of the GitHub pull request and the real GPU execution results have **not** been independently verified. It orchestrates the repository's training, calibration, evaluation, and paired-analysis modules without duplicating the sparse-attention implementation.

## Choose an execution route

| Route | Required input | Workflow |
|---|---|---|
| `original_v4` **(default)** | Teammate's original `best.pt` or `Sparge_Project_Backup.zip` | Verify original model and calibration, perform integration checks, then run the original official-test launcher |
| `retrain` | Access to ImageNet pretrained weights and enough GPU time | Train a new dense model, recalibrate on validation, run complete validation, prepare a new experiment protocol, and then evaluate the official test split |

**Use `original_v4` if you have obtained the original checkpoint.** The earlier `RUN_NEXT_COLAB.md` specifies `outputs/dense_full_pilot/best.pt` inside the backup ZIP. Cloning the GitHub repository does not automatically retrieve model weights.

The retraining workflow also requires the companion `scripts/run_reviewed_test.py` and `tests/test_reviewed_runner.py` from the supporting bundle to be committed and reviewed before conducting a new official test. It does not bypass the original V4 fingerprint checks.

## Execution rules

- Select a GPU runtime in Colab. Keep temporary code and downloaded data under `/content`; persist logs, checkpoints, and results to the mounted Google Drive.
- Execute cells in order. Training and official testing are **disabled by default** and require explicit opt-in.
- Freeze the checkpoint, preprocessing, attention-layer set, calibration, and full Git commit before testing. Do not retune on official test results.
- The experiment-freeze documentation requires real team approval; blank approval fields below are intentional.
- The official evaluation does **not** train models, benchmark training speed, or measure GPU inference latency or memory. The prototype still performs dense attention matrix multiplications.
- Validation/calibration utilities may initialize dataset objects or download source archives, but they do not select hyperparameters using test predictions.


## Block 1 — Configure the experiment

Set the paths and execution route before running anything else. With the original `best.pt` available, keep `ROUTE = "original_v4"`. The notebook fetches PR #1 by default; use `CHECKOUT_COMMIT` after the team agrees on a specific full 40-character Git SHA. Do not put access tokens in the notebook.

The retraining hyperparameters are preliminary pilot settings, not guaranteed best values. In the training code, `--resume` is a weight-initialization warm start rather than an optimizer/scheduler-state resume.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, uuid, re, shutil, zipfile, tempfile, signal

ROUTE = "original_v4"  # "original_v4" or "retrain"
REPO_URL = "https://github.com/TharunSwaminathan/deit-sparse-attention.git"
CODE_REF = "refs/pull/1/head"  # Or use refs/heads/jiong/final-evaluation for your branch
CHECKOUT_COMMIT = ""  # Provide a team-approved full 40-character SHA when available
USE_GITHUB_SECRET = False  # For private repos, use a GITHUB_TOKEN stored in Colab Secrets
REPO_DIR = Path("/content/deit-sparse-attention")
PROJECT_DRIVE = Path("/content/drive/MyDrive/DeiT_Final_Project")
ORIGINAL_PT = PROJECT_DRIVE / "checkpoints" / "best.pt"
BACKUP_ZIP = PROJECT_DRIVE / "Sparge_Project_Backup.zip"
DATA_ROOT = Path("/content/pets_data")

RUN_VALIDATION_PILOT = True
RUN_HEAD_TRAINING = False
RUN_FULL_FINETUNING = False
RUN_RECALIBRATION = False
RUN_FULL_VALIDATION = False
RETRAIN_NAME = "dense_retrain_001"
CALIBRATION_NAME = "calibration_001"
VALIDATION_NAME = "full_validation_001"
OFFICIAL_RUN_NAME = "official_test_v4_001"  # Use a new run name for a newly trained checkpoint
HEAD_EPOCHS, HEAD_LR = 8, 1e-3
FULL_EPOCHS, FULL_LR = 5, 1e-5
FINAL_DENSE_STAGE = "best_val_loss"  # or "head" / "full"
CALIBRATION_IMAGES = 64
P_GRID = [0.65, 0.85, 0.87, 0.95, 0.97, 0.98]
TOP_K_GRID = [12, 16, 32]
SIMILARITY_GRID = [0.45, 0.50, 0.55]
LAYERS = [3, 6, 9]  # zero-indexed; retained for consistency, not claimed optimal
TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE, WORKERS, SEED = 32, 16, 0, 42

assert ROUTE in {"original_v4", "retrain"}
assert not CHECKOUT_COMMIT or re.fullmatch(r"[0-9a-fA-F]{40}", CHECKOUT_COMMIT)
for name in (RETRAIN_NAME, CALIBRATION_NAME, VALIDATION_NAME, OFFICIAL_RUN_NAME):
    assert re.fullmatch(r"[A-Za-z0-9_-]+", name), "Use only letters, digits, underscores, or hyphens in experiment names."
CONFIGURATIONS = ["dense", "unmasked"] + [f"{v}_{s}" for s in (25, 50) for v in "ABCD"]
EXPECTED_V4_PT_SHA = "ca12f33e158e30c08279ff8be5bd5b2a69be75e688a01eee59195fc3f198cb06"
EXPECTED_V4_CAL_SHA = "c04ebcc0534cf814f660be6357479d44c948a65f6daa0e8d631274915201d0e4"
EXPECTED_SPLIT_SHA = "a0666d0cb771f7ce8ee0f45600d099b68d412bf8bf76cfeddca25c7fbc3b0b33"
print("Route:", ROUTE, "; source revision:", CHECKOUT_COMMIT or CODE_REF)


## Block 2 — Mount Google Drive and create session directories

Mount Drive and create a unique directory for logs and environment records. Only the **parent** of the official-results directory is created now; the official launcher requires its final output directory to be absent before execution.

Mounting Drive does not automatically back up `/content`. This notebook explicitly directs persistent artifacts to Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SESSION_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
SESSION_DIR = PROJECT_DRIVE / "sessions" / SESSION_TAG
SESSION_DIR.mkdir(parents=True, exist_ok=False)
RETRAIN_DIR = PROJECT_DRIVE / "retraining" / RETRAIN_NAME
OFFICIAL_DIR = PROJECT_DRIVE / "results" / OFFICIAL_RUN_NAME
DERIVED_DIR = PROJECT_DRIVE / "derived" / OFFICIAL_RUN_NAME
OFFICIAL_DIR.parent.mkdir(parents=True, exist_ok=True)
LOCAL_INPUTS = Path("/content/deit_inputs")
LOCAL_INPUTS.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def raw_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def canonical_sha(path):
    value = json.loads(Path(path).read_text(encoding="utf-8"))
    data = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    return hashlib.sha256(data).hexdigest()

def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")

def run_logged(argv, label, cwd=None):
    logfile = SESSION_DIR / (label + "_" + uuid.uuid4().hex[:6] + ".log")
    argv = [str(x) for x in argv]
    with logfile.open("w", encoding="utf-8") as out:
        process = subprocess.Popen(argv, cwd=cwd or REPO_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                out.write(line)
                out.flush()
            returncode = process.wait()
        except BaseException:
            for sig, grace in [(signal.SIGINT,10), (signal.SIGTERM,5), (signal.SIGKILL,3)]:
                try:
                    os.killpg(process.pid, sig)
                except ProcessLookupError:
                    break
                try:
                    process.wait(timeout=grace)
                    break
                except subprocess.TimeoutExpired:
                    continue
            raise
    if returncode:
        raise RuntimeError(f"{label} failed (exit={returncode}). Log preserved: {logfile}")
    return logfile

print("Session logs:", SESSION_DIR)
print("Official output directory (not yet created):", OFFICIAL_DIR)


## Block 3 — Retrieve and pin the GitHub code revision

A normal `git clone` may not include unmerged changes from PR #1. This cell explicitly fetches the PR reference and checks out the resolved full commit SHA in detached-HEAD mode.

An existing directory with local modifications causes the cell to stop rather than overwrite work. It does not merge the PR, push commits, or approve a GitHub pull request. A private repository requires authorized credentials.


In [ ]:
def git_call(args, cwd=None):
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    token = None
    with tempfile.TemporaryDirectory(prefix="deit_git_auth_") as authdir:
        if USE_GITHUB_SECRET:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            if not token:
                raise RuntimeError("No valid GITHUB_TOKEN found in Colab Secrets.")
            helper = Path(authdir) / "askpass.py"
            helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['DEIT_GITHUB_TOKEN'])\n")
            helper.chmod(0o700)
            env.update(GIT_ASKPASS=str(helper), DEIT_GITHUB_TOKEN=token)
        result = subprocess.run(["git", *args], cwd=cwd, env=env, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        message = result.stdout + result.stderr
        if token:
            message = message.replace(token, "[REDACTED]")
        if result.returncode:
            raise RuntimeError(message + "\nCheck repository permissions and the requested ref; do not embed tokens in URLs.")
        return result.stdout.strip()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError("Target directory exists but is not a Git checkout. Choose a new REPO_DIR.")
    origin = git_call(["remote", "get-url", "origin"], REPO_DIR)
    if origin.rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("The existing checkout has a different origin URL.")
    if git_call(["status", "--porcelain"], REPO_DIR):
        raise RuntimeError("Uncommitted local changes detected. Save or commit them first; this notebook will not overwrite changes.")
else:
    git_call(["clone", "--no-checkout", REPO_URL, str(REPO_DIR)])

git_call(["fetch", "origin", CHECKOUT_COMMIT or CODE_REF], REPO_DIR)
REVISION = git_call(["rev-parse", "FETCH_HEAD"], REPO_DIR)
if CHECKOUT_COMMIT and REVISION.lower() != CHECKOUT_COMMIT.lower():
    raise RuntimeError("Fetched commit does not match the specified full SHA.")
if any(name == "sparse_pet" or name.startswith("sparse_pet.") for name in sys.modules):
    old_head = git_call(["rev-parse", "HEAD"], REPO_DIR)
    if old_head != REVISION:
        raise RuntimeError("The kernel previously imported sparse_pet from another revision. Restart the runtime before switching revisions.")
git_call(["checkout", "--detach", REVISION], REPO_DIR)
assert not git_call(["status", "--porcelain"], REPO_DIR)
MANIFEST = REPO_DIR / "outputs" / "split_manifest.json"
for required in ["requirements.txt", "sparse_pet/train.py", "sparse_pet/evaluate.py",
                 "sparse_pet/calibrate.py", "sparse_pet/paired_analysis.py", "scripts/run_official_test.py"]:
    if not (REPO_DIR / required).is_file():
        raise FileNotFoundError(f"Required path {required} is missing from this commit; check the PR/ref.")
if not MANIFEST.is_file() or canonical_sha(MANIFEST) != EXPECTED_SPLIT_SHA:
    raise RuntimeError("The split manifest is missing or differs from the project snapshot. Investigate rather than regenerating it.")
save_json(SESSION_DIR / "source.json", {"repo":REPO_URL, "ref":CODE_REF, "revision":REVISION})
print("Checked out full SHA:", REVISION)
print("Working tree is clean. Formal test requires team approval of this revision.")
if ROUTE == "retrain" and not (REPO_DIR / "scripts/run_reviewed_test.py").is_file():
    print("The retraining route also requires run_reviewed_test.py committed to the experiment branch. Training may start now, but commit the runner before full validation.")


## Block 4 — Install project dependencies

Install the repository requirements and visualization packages while keeping the current Colab-provided `torch` / `torchvision` versions constrained together. If the dependencies are incompatible, stop and select a compatible GPU runtime instead of silently substituting CPU execution.


In [ ]:
from importlib import metadata
installed = {}
for package in ["torch", "torchvision"]:
    try:
        installed[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing {package}. Select a Colab GPU runtime with compatible PyTorch/CUDA packages.")
constraints = LOCAL_INPUTS / "colab_torch_constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in installed.items()))
run_logged([sys.executable, "-m", "pip", "install", "-r", REPO_DIR / "requirements.txt",
            "-c", constraints, "pandas", "matplotlib"], "install")
print("Preserving installed GPU base dependencies:", installed)


## Block 5 — Verify the GPU and record the environment

Confirm CUDA availability and record exact Python, PyTorch, torchvision, timm, NumPy, scikit-learn, GPU, and preprocessing settings. These records describe this session and must not be confused with the teammate's earlier T4 benchmark.


In [ ]:
import torch, torchvision, timm, numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is unavailable. Select a GPU in Colab runtime settings and retry.")
RUNTIME = {"python":sys.version, "torch":torch.__version__, "torchvision":torchvision.__version__,
           "timm":timm.__version__, "numpy":np.__version__, "sklearn":sklearn.__version__,
           "gpu":torch.cuda.get_device_name(0), "cuda_runtime":torch.version.cuda,
           "code_revision":REVISION, "recorded_utc":utc_now()}
save_json(SESSION_DIR / "environment.json", RUNTIME)
from timm.data import resolve_model_data_config, create_transform
metadata_model = timm.create_model("deit_tiny_patch16_224.fb_in1k", pretrained=False, num_classes=37)
eval_data_config = resolve_model_data_config(metadata_model)
RUNTIME["eval_data_config"] = eval_data_config
RUNTIME["eval_transform"] = repr(create_transform(**eval_data_config, is_training=False))
del metadata_model
save_json(SESSION_DIR / "environment.json", RUNTIME)
save_json(SESSION_DIR / "package_versions.json", {d.metadata["Name"]:d.version for d in metadata.distributions() if d.metadata.get("Name")})
print(json.dumps(RUNTIME, indent=2))


## Block 6 — Run the unit tests

Run the tests from the exact checked-out source revision. The teammate reported 56 passing tests, but this notebook does not assume that number: use the actual output from this run. Stop and investigate failures before proceeding.


In [ ]:
TESTS_OK = False
run_logged([sys.executable, "-m", "pytest", "-q", "tests"], "pytest")
assert not git_call(["status", "--porcelain"], REPO_DIR), "The working tree changed after tests; investigate."
TESTS_OK = True
TESTED_REVISION = git_call(["rev-parse", "HEAD"], REPO_DIR)
print("Pytest completed successfully.")


## Block 7 — Recover and verify the original V4 checkpoint

Read `best.pt` from the configured Drive location; if it is absent, look inside `Sparge_Project_Backup.zip` for `outputs/dense_full_pilot/best.pt`. Only the weight file is extracted—older source files and configurations are never copied over the current Git checkout.

For the original route, verify the checkpoint against the expected V4 SHA-256. A different hash is a blocker, not permission to silently substitute another model.


In [ ]:
CHECKPOINT = None
if ROUTE == "original_v4":
    checkpoint_local = LOCAL_INPUTS / "original_v4_best.pt"
    if ORIGINAL_PT.is_file():
        if raw_sha(ORIGINAL_PT) != EXPECTED_V4_PT_SHA:
            raise RuntimeError("The supplied best.pt does not match the original V4 checkpoint. Verify its provenance; it will not be treated as a new model automatically.")
        shutil.copy2(ORIGINAL_PT, checkpoint_local)
    elif BACKUP_ZIP.is_file():
        with zipfile.ZipFile(BACKUP_ZIP) as backup:
            candidates = [n for n in backup.namelist() if n.replace("\\", "/").endswith("outputs/dense_full_pilot/best.pt")]
            if len(candidates) != 1:
                raise RuntimeError(f"Found {len(candidates)} candidate weights in the backup. Inspect manually and place the correct checkpoint at ORIGINAL_PT.")
            with backup.open(candidates[0]) as source, checkpoint_local.open("wb") as dest:
                shutil.copyfileobj(source, dest)
        if raw_sha(checkpoint_local) != EXPECTED_V4_PT_SHA:
            raise RuntimeError("The checkpoint SHA-256 inside the backup does not match. Preserve the backup and verify it with the teammate.")
        ORIGINAL_PT.parent.mkdir(parents=True, exist_ok=True)
        if not ORIGINAL_PT.exists():
            shutil.copy2(checkpoint_local, ORIGINAL_PT)
    else:
        raise FileNotFoundError(f"Neither {ORIGINAL_PT} nor {BACKUP_ZIP} was found. Request Sparge_Project_Backup.zip; select retrain only if recovery is impossible.")
    CHECKPOINT = checkpoint_local
    print("Original V4 checkpoint verified:", raw_sha(CHECKPOINT))
else:
    print("Retraining route selected. Run the training blocks next; the old V4 checkpoint and results will not be reused.")


## Block 8 — Fallback only: train the classification head

**Skip this block when `ROUTE = "original_v4"`.** Under `ROUTE = "retrain"`, explicitly enable `RUN_HEAD_TRAINING` to train the classifier while keeping the pretrained backbone frozen. The train/validation split is read from the existing manifest; the official test split is not used for model selection.

Outputs are written to Drive. Running the training script again without carefully handling previous outputs is not a reliable continuation of optimizer state.


In [ ]:
def train_stage(stage, enabled, resume=None):
    folder = RETRAIN_DIR / stage
    record_path = folder / "stage_record.json"
    settings = {"mode":stage, "epochs":HEAD_EPOCHS if stage == "head" else FULL_EPOCHS,
                "lr":HEAD_LR if stage == "head" else FULL_LR, "batch_size":TRAIN_BATCH_SIZE,
                "seed":SEED, "warmup_epochs":1, "weight_decay":0.05,
                "manifest_canonical_sha256":canonical_sha(MANIFEST),
                "resume_sha256":raw_sha(resume) if resume else None}
    if folder.exists():
        if not record_path.is_file():
            raise RuntimeError(f"{folder} exists without a complete record. Preserve it and change RETRAIN_NAME.")
        record = json.loads(record_path.read_text())
        if record.get("status") != "complete" or record.get("settings") != settings:
            raise RuntimeError("Existing training is incomplete or has different settings. Do not overwrite; choose another experiment name.")
        if raw_sha(folder / "best.pt") != record["checkpoint_sha256"]:
            raise RuntimeError("Existing training weights do not match the recorded checkpoint hash.")
        print("Reusing completed training:", folder)
        return folder / "best.pt"
    if not enabled:
        print(f"Training for {stage} is disabled; enable the corresponding Block 1 switch if needed.")
        return None
    folder.mkdir(parents=True, exist_ok=False)
    record = {"status":"running", "settings":settings, "code_revision":REVISION,
              "started_utc":utc_now(), "runtime":RUNTIME}
    save_json(record_path, record)
    command = [sys.executable, "-m", "sparse_pet.train", "--mode", stage,
               "--epochs", str(settings["epochs"]), "--lr", str(settings["lr"]),
               "--batch-size", str(TRAIN_BATCH_SIZE), "--seed", str(SEED), "--workers", str(WORKERS),
               "--warmup-epochs", "1", "--weight-decay", "0.05", "--data-root", str(DATA_ROOT),
               "--manifest", str(MANIFEST), "--download", "--output", str(folder)]
    if resume:
        command += ["--resume", str(resume)]
    try:
        run_logged(command, "train_" + stage)
        for name in ["best.pt", "history.json", "summary.json"]:
            assert (folder / name).is_file()
        record.update(status="complete", checkpoint_sha256=raw_sha(folder / "best.pt"))
    except BaseException as error:
        record.update(status="failed", error=repr(error))
        raise
    finally:
        record["finished_utc"] = utc_now()
        save_json(record_path, record)
    return folder / "best.pt"

HEAD_CHECKPOINT = None
if ROUTE == "retrain":
    HEAD_CHECKPOINT = train_stage("head", RUN_HEAD_TRAINING)
    if HEAD_CHECKPOINT:
        display(pd.DataFrame(json.loads((HEAD_CHECKPOINT.parent / "history.json").read_text())))
else:
    print("Skipped: original V4 checkpoint is in use.")


## Block 9 — Fallback only: optional full fine-tuning and checkpoint selection

**Skip this block for the original V4 route.** If retraining, inspect the head-only validation curves before deciding whether to enable full fine-tuning. Full fine-tuning updates the Transformer backbone as well as the classification head.

Choose one final dense checkpoint using validation performance, and use that single checkpoint consistently for all sparse variants. A new checkpoint cannot inherit the original V4 calibration or validation results.


In [ ]:
if ROUTE == "retrain":
    if HEAD_CHECKPOINT is None:
        raise RuntimeError("Finish head training in Block 8 or restore its complete output first.")
    FULL_CHECKPOINT = train_stage("full", RUN_FULL_FINETUNING, resume=HEAD_CHECKPOINT)
    candidates = {"head":HEAD_CHECKPOINT}
    if FULL_CHECKPOINT:
        candidates["full"] = FULL_CHECKPOINT
        display(pd.DataFrame(json.loads((FULL_CHECKPOINT.parent / "history.json").read_text())))
    summaries = {name:json.loads((path.parent / "summary.json").read_text()) for name,path in candidates.items()}
    if FINAL_DENSE_STAGE == "best_val_loss":
        stage = min(candidates, key=lambda name:float(summaries[name]["best_val_loss"]))
    else:
        if FINAL_DENSE_STAGE not in candidates:
            raise RuntimeError("Selected training stage does not have a complete result.")
        stage = FINAL_DENSE_STAGE
    CHECKPOINT = candidates[stage]
    save_json(RETRAIN_DIR / "dense_selection.json", {"criterion":FINAL_DENSE_STAGE, "stage":stage,
              "checkpoint":str(CHECKPOINT), "checkpoint_sha256":raw_sha(CHECKPOINT), "summaries":summaries})
    print("Selected new dense checkpoint:", CHECKPOINT)
    print("SHA-256:", raw_sha(CHECKPOINT))
else:
    print("Skipped: original V4 checkpoint is fixed.")


## Block 10 — Verify attention equivalence with the real checkpoint

Load the selected model and run the repository's attention smoke test on synthetic inputs (without evaluating test images). Check that native dense, unmasked wrapper, and all-kept attention outputs are numerically consistent and that sparse outputs remain finite.

This catches implementation mismatches before proceeding to official evaluation.


In [ ]:
SMOKE_OK = False
assert TESTS_OK and CHECKPOINT is not None and CHECKPOINT.is_file()
assert git_call(["rev-parse", "HEAD"], REPO_DIR) == TESTED_REVISION
CHECKPOINT_SHA = raw_sha(CHECKPOINT)
run_logged([sys.executable, "-m", "sparse_pet.smoke_test", "--checkpoint", CHECKPOINT,
            "--layers", ",".join(map(str, LAYERS)), "--device", "cuda",
            "--output", SESSION_DIR / "smoke.json"], "checkpoint_smoke")
assert json.loads((SESSION_DIR / "smoke.json").read_text())["pass_equivalence"] is True
SMOKE_OK = True


## Block 11 — Verify frozen V4 calibration or recalibrate a new model

For the original V4 route, read and verify the existing calibration file. **Do not rerun the older v3 grid search.** For a newly trained checkpoint, calibration must be performed using validation images and then reviewed before the test run.

Display the eight selected configurations, realized calibration sparsities, and any feasibility or mechanism-activity warnings. This cell must not use official test results for hyperparameter selection.


In [ ]:
if ROUTE == "original_v4":
    CALIBRATION = REPO_DIR / "outputs" / "calibration_64_v4.json"
    if canonical_sha(CALIBRATION) != EXPECTED_V4_CAL_SHA:
        raise RuntimeError("Original V4 calibration fingerprint mismatch. Verify the code revision and calibration file.")
else:
    if not (REPO_DIR / "scripts/run_reviewed_test.py").is_file():
        raise RuntimeError("Before recalibration, commit the companion runner and notebook to the experiment branch and check out that revision. Preserve any trained checkpoint.")
    git_call(["ls-files", "--error-unmatch", "scripts/run_reviewed_test.py"], REPO_DIR)
    assert not git_call(["status", "--porcelain"], REPO_DIR)
    CALIBRATION = RETRAIN_DIR / f"{CALIBRATION_NAME}.json"
    calibration_record = RETRAIN_DIR / f"{CALIBRATION_NAME}_record.json"
    grid_record = {"checkpoint_sha256":CHECKPOINT_SHA, "p":P_GRID, "top_k":TOP_K_GRID,
                   "similarity":SIMILARITY_GRID, "images":CALIBRATION_IMAGES, "layers":LAYERS,
                   "code_revision":REVISION, "manifest_canonical_sha256":canonical_sha(MANIFEST)}
    if CALIBRATION.exists():
        old_record = json.loads(calibration_record.read_text()) if calibration_record.exists() else {}
        if old_record.get("settings") != grid_record or old_record.get("status") != "complete":
            raise RuntimeError("Existing calibration provenance or settings differ, or calibration is incomplete. Change CALIBRATION_NAME.")
        assert canonical_sha(CALIBRATION) == old_record["calibration_canonical_sha256"]
        print("Reusing completed calibration for the new model:", CALIBRATION)
    elif RUN_RECALIBRATION:
        run_logged([sys.executable, "-m", "sparse_pet.calibrate", "--checkpoint", CHECKPOINT,
                    "--data-root", DATA_ROOT, "--manifest", MANIFEST, "--download",
                    "--layers", ",".join(map(str, LAYERS)), "--batch-size", str(EVAL_BATCH_SIZE),
                    "--workers", str(WORKERS), "--images", str(CALIBRATION_IMAGES),
                    "--targets", "0.25", "0.50", "--p-grid", *map(str,P_GRID),
                    "--top-k-grid", *map(str,TOP_K_GRID), "--similarity-grid", *map(str,SIMILARITY_GRID),
                    "--output", CALIBRATION], "calibration")
        save_json(calibration_record, {"status":"complete", "settings":grid_record,
                  "calibration_canonical_sha256":canonical_sha(CALIBRATION)})
    else:
        raise RuntimeError("The new model has not been calibrated. Verify settings, enable RUN_RECALIBRATION, and rerun this block.")

calibration = json.loads(CALIBRATION.read_text())
assert calibration["checkpoint_sha256"] == CHECKPOINT_SHA
assert calibration["layer_indices"] == LAYERS
assert set(calibration["selected"]) == set(CONFIGURATIONS[2:])
CALIBRATION_SHA = canonical_sha(CALIBRATION)
display(pd.DataFrame([{"configuration":label, **row} for label,row in calibration["selected"].items()])[
    ["configuration", "p", "top_k", "similarity_threshold", "realized_sparsity", "feasible_within_tolerance", "mechanisms_active"]])
for warning in calibration.get("calibration_warnings", []):
    print("Calibration note:", warning)


## Block 12 — Small validation-set integration pilot

Run Dense, Unmasked, and A_50 on the same 32 validation images to confirm that the real dataset, checkpoint, evaluation code, and per-image prediction export work together.

Check aligned image IDs and labels; Dense and Unmasked should have no prediction disagreements. This is a pipeline check, **not** final evidence of classification accuracy.


In [ ]:
PILOT_OK = False
def evaluation_command(label, split, directory, max_images=None):
    command = [sys.executable, "-m", "sparse_pet.evaluate", "--checkpoint", str(CHECKPOINT),
               "--data-root", str(DATA_ROOT), "--manifest", str(MANIFEST), "--split", split,
               "--frozen-config", str(CALIBRATION), "--batch-size", str(EVAL_BATCH_SIZE),
               "--workers", str(WORKERS), "--download"]
    command += ["--variant", label] if label in ("dense", "unmasked") else ["--label", label]
    if max_images is not None:
        assert split == "val"
        command += ["--max-images", str(max_images)]
    return command + ["--output", str(directory / f"{label}.json"),
                      "--predictions-output", str(directory / f"{label}.csv")]

if RUN_VALIDATION_PILOT:
    PILOT_DIR = SESSION_DIR / ("validation_pilot_" + uuid.uuid4().hex[:6])
    for label in ["dense", "unmasked", "A_50"]:
        run_logged(evaluation_command(label, "val", PILOT_DIR, 32), "pilot_" + label)
    sys.path.insert(0, str(REPO_DIR))
    from sparse_pet.paired_analysis import read_run
    dense_pilot, _ = read_run(PILOT_DIR, "dense", "val")
    allowed = {f"trainval:{i}" for i in json.loads(MANIFEST.read_text())["val_indices"]}
    assert len(dense_pilot) == 32 and set(dense_pilot).issubset(allowed)
    for label in ["unmasked", "A_50"]:
        rows, _ = read_run(PILOT_DIR, label, "val")
        assert set(rows) == set(dense_pilot)
        assert all(rows[k][0] == dense_pilot[k][0] for k in rows)
        differences = sum(rows[k][1] != dense_pilot[k][1] for k in rows)
        print(label, "prediction disagreements on 32 validation images:", differences)
        if label == "unmasked" and differences:
            raise RuntimeError("Dense and Unmasked controls disagree. Investigate numerics and implementation before official testing.")
    PILOT_OK = True
    print("Validation pilot passed; this is not an official test result.")
else:
    print("Validation pilot was not executed. Complete it before the official test.")


## Block 13 — Full validation for a newly retrained model

**Skip for `original_v4`.** If a new checkpoint was trained, evaluate all ten configurations on the 736 validation images and retain the per-image outputs and provenance records. Do not reuse the teammate's former 93.75% accuracy, calibration table, or latency measurements as measurements for the new model.

The supporting reviewed-test runner must first be added to a reviewed Git commit, and `RUN_FULL_VALIDATION` must be enabled when you are ready.


In [ ]:
VALIDATION_DIR = RETRAIN_DIR / VALIDATION_NAME
if ROUTE == "retrain":
    sys.path.insert(0, str(REPO_DIR))
    from sparse_pet.paired_analysis import read_run
    new_runner = REPO_DIR / "scripts/run_reviewed_test.py"
    if not new_runner.is_file():
        raise RuntimeError("Commit the companion runner, tests, and notebook to the experiment branch, check out the new commit, and rerun environment checks.")
    assert not git_call(["status", "--porcelain"], REPO_DIR)
    assert git_call(["rev-parse", "HEAD"], REPO_DIR) == REVISION
    validation_record_path = VALIDATION_DIR / "validation_provenance.json"
    if VALIDATION_DIR.exists():
        if not validation_record_path.is_file():
            raise RuntimeError("Existing validation directory is incomplete. Preserve it and change VALIDATION_NAME.")
        record = json.loads(validation_record_path.read_text())
        assert record.get("status") == "complete", "Validation is incomplete; do not overwrite it."
        assert record["code_revision"] == REVISION and record["checkpoint_sha256"] == CHECKPOINT_SHA
        assert record["calibration_canonical_sha256"] == CALIBRATION_SHA
        assert record["manifest_canonical_sha256"] == canonical_sha(MANIFEST)
        for filename, fingerprint in record["output_sha256"].items():
            assert raw_sha(VALIDATION_DIR / filename) == fingerprint
        print("Reusing completed full validation.")
    elif not RUN_FULL_VALIDATION:
        raise RuntimeError("Inspect calibration results, enable RUN_FULL_VALIDATION, and rerun this block.")
    else:
        VALIDATION_DIR.mkdir(parents=True, exist_ok=False)
        record = {"schema_version":1, "status":"running", "code_revision":REVISION,
                  "checkpoint_sha256":CHECKPOINT_SHA, "calibration_canonical_sha256":CALIBRATION_SHA,
                  "manifest_canonical_sha256":canonical_sha(MANIFEST), "layers":LAYERS,
                  "batch_size":EVAL_BATCH_SIZE, "workers":WORKERS, "n":736,
                  "configurations":CONFIGURATIONS, "output_sha256":{}, "started_utc":utc_now(), "runtime":RUNTIME}
        save_json(validation_record_path, record)
        try:
            expected_ids = {f"trainval:{i}" for i in json.loads(MANIFEST.read_text())["val_indices"]}
            reference_rows = None
            for label in CONFIGURATIONS:
                assert raw_sha(CHECKPOINT) == CHECKPOINT_SHA
                assert canonical_sha(CALIBRATION) == CALIBRATION_SHA
                assert canonical_sha(MANIFEST) == EXPECTED_SPLIT_SHA
                assert git_call(["rev-parse", "HEAD"], REPO_DIR) == REVISION
                assert not git_call(["status", "--porcelain"], REPO_DIR)
                run_logged(evaluation_command(label, "val", VALIDATION_DIR), "full_val_" + label)
                rows, fingerprints = read_run(VALIDATION_DIR, label, "val")
                assert set(rows) == expected_ids and len(rows) == 736
                if reference_rows is None:
                    reference_rows = rows
                assert all(rows[k][0] == reference_rows[k][0] for k in rows)
                record["output_sha256"].update(fingerprints)
                save_json(validation_record_path, record)
            assert raw_sha(CHECKPOINT) == CHECKPOINT_SHA
            assert canonical_sha(CALIBRATION) == CALIBRATION_SHA
            assert canonical_sha(MANIFEST) == EXPECTED_SPLIT_SHA
            assert git_call(["rev-parse", "HEAD"], REPO_DIR) == REVISION
            assert not git_call(["status", "--porcelain"], REPO_DIR)
            record["status"] = "complete"
        except BaseException as error:
            record.update(status="failed", error=repr(error))
            raise
        finally:
            record["finished_utc"] = utc_now()
            save_json(validation_record_path, record)
    rows = []
    for label in CONFIGURATIONS:
        result = json.loads((VALIDATION_DIR / f"{label}.json").read_text())
        rows.append({"configuration":label, "val_accuracy_pct":100*result["scores"]["accuracy"],
                     "val_macro_f1":result["scores"]["macro_f1"],
                     "val_sparsity_pct":100*result["diagnostics"].get("realized_sparsity",0.0)})
    display(pd.DataFrame(rows))
else:
    print("Original V4 already has full validation results; this notebook does not repeat calibration or full validation.")


## Block 14 — Review the experiment-freeze materials

For original V4, display `docs/EXPERIMENT_FREEZE.md` and the actual checked-out Git SHA. For a new checkpoint, construct a separate reviewable protocol tied to checkpoint, calibration, split manifest, validation evidence, code revision, and runner fingerprint.

Preparing or displaying this record is **not** the same as obtaining team approval. The experiment must not proceed until everyone agrees on the final settings.


In [ ]:
PROTOCOL = RETRAIN_DIR / f"{CALIBRATION_NAME}_{VALIDATION_NAME}_protocol.json"
if ROUTE == "original_v4":
    print((REPO_DIR / "docs/EXPERIMENT_FREEZE.md").read_text())
    print("Actual checked-out SHA:", REVISION, "The abbreviated SHA in old documentation is not automatically approved.")
else:
    if not PROTOCOL.exists():
        run_logged([sys.executable, "scripts/run_reviewed_test.py", "--prepare",
                    "--checkpoint", CHECKPOINT, "--calibration", CALIBRATION, "--manifest", MANIFEST,
                    "--validation-dir", VALIDATION_DIR, "--protocol", PROTOCOL], "prepare_protocol")
    print("New protocol:", PROTOCOL)
    print("Protocol SHA-256 pending review:", raw_sha(PROTOCOL))
    print(PROTOCOL.read_text())


## Block 15 — Run the official-test preflight (dry run)

Invoke the launcher without `--execute`. It checks configured input fingerprints, Git state, and planned commands, but **does not** perform test inference or verify that the full dataset and GPU environment work.

The earlier unit tests, checkpoint smoke test, and validation pilot provide complementary checks.


In [ ]:
PREFLIGHT_OK = False
if ROUTE == "original_v4":
    OFFICIAL_COMMAND = [sys.executable, "scripts/run_official_test.py",
                       "--checkpoint", str(CHECKPOINT), "--data-root", str(DATA_ROOT),
                       "--output-dir", str(OFFICIAL_DIR)]
else:
    OFFICIAL_COMMAND = [sys.executable, "scripts/run_reviewed_test.py", "--protocol", str(PROTOCOL),
                       "--data-root", str(DATA_ROOT), "--output-dir", str(OFFICIAL_DIR)]
run_logged(OFFICIAL_COMMAND, "official_preflight")
PREFLIGHT_OK = True
print("Preflight completed; no official test inference has run.")


## Block 16 — Enter real team approval information

The teammate's message asking for review is not itself team approval. Complete the fields below **only after** the team reviews and freezes the exact source revision and experiment inputs.

- `APPROVED_REVISION`: the team's approved full 40-character commit SHA.
- `APPROVAL_REFERENCE`: a genuine dated approval record, such as a team issue or message reference.
- `APPROVED_PROTOCOL_SHA256`: required only for the retraining route.
- `RUN_OFFICIAL_TEST`: change from `False` to `True` only when ready to execute the held-out test once.


In [ ]:
APPROVED_REVISION = ""
APPROVAL_REFERENCE = ""
APPROVED_PROTOCOL_SHA256 = ""  # required only for the retraining route
RUN_OFFICIAL_TEST = False
print("Official test switch:", RUN_OFFICIAL_TEST)


## Block 17 — Run the official held-out evaluation

Execute the approved launcher with ten configurations and the paired prediction analysis. This phase freezes model weights and uses the complete 3,669-image official test set with batch size 16 and FP32 inference.

No `--max-images` or runtime benchmark options are passed. Results are saved directly to Drive in a **new** output directory. On failure, retain the partial outputs and diagnostics; do not overwrite or combine separate runs.


In [ ]:
if not RUN_OFFICIAL_TEST:
    print("Official test is disabled. Preparation steps can be run independently.")
else:
    assert TESTS_OK and SMOKE_OK and PILOT_OK and PREFLIGHT_OK
    head_now = git_call(["rev-parse", "HEAD"], REPO_DIR)
    assert re.fullmatch(r"[0-9a-f]{40}", APPROVED_REVISION)
    assert head_now == APPROVED_REVISION == TESTED_REVISION == REVISION
    assert not git_call(["status", "--porcelain"], REPO_DIR)
    assert APPROVAL_REFERENCE.strip(), "Provide a genuine team approval reference."
    assert not OFFICIAL_DIR.exists(), "Official output directory already exists; overwriting or combining runs is prohibited."
    assert raw_sha(CHECKPOINT) == CHECKPOINT_SHA
    assert canonical_sha(CALIBRATION) == CALIBRATION_SHA
    assert canonical_sha(MANIFEST) == EXPECTED_SPLIT_SHA
    execution = OFFICIAL_COMMAND + ["--execute", "--download", "--approved-revision", APPROVED_REVISION,
                                    "--approval-reference", APPROVAL_REFERENCE]
    if ROUTE == "retrain":
        assert re.fullmatch(r"[0-9a-f]{64}", APPROVED_PROTOCOL_SHA256)
        assert raw_sha(PROTOCOL) == APPROVED_PROTOCOL_SHA256
        execution += ["--approved-protocol-sha256", APPROVED_PROTOCOL_SHA256]
    run_logged(execution, "official_execution")
    print("Launcher returned successfully. Validate its outputs without changing test settings.")


## Block 18 — Validate the official results

Verify that all 22 expected raw output files exist, all ten prediction sets contain 3,669 unique aligned test-image IDs, aggregate accuracy agrees with exported predictions, and the stored fingerprints and paired-analysis inputs match.

Investigate any Dense/Unmasked prediction disagreement before drawing conclusions. This cell does not rerun model inference.


In [ ]:
OUTPUTS_OK = False
RESULTS = {}
PAIRED = None
if not (OFFICIAL_DIR / "run_provenance.json").is_file():
    print("No official test results yet; skipping analysis.")
else:
    provenance = json.loads((OFFICIAL_DIR / "run_provenance.json").read_text())
    assert provenance["status"] == "complete", "This run is incomplete or failed; it cannot be used for the final results table."
    assert provenance["revision"] == REVISION, "These results belong to a different source revision and must not be mixed."
    assert raw_sha(CHECKPOINT) == CHECKPOINT_SHA and canonical_sha(CALIBRATION) == CALIBRATION_SHA
    assert canonical_sha(MANIFEST) == EXPECTED_SPLIT_SHA
    expected_hashes = {"checkpoint_sha256":CHECKPOINT_SHA, "calibration_canonical_sha256":CALIBRATION_SHA,
                       "manifest_canonical_sha256":EXPECTED_SPLIT_SHA}
    if ROUTE == "original_v4":
        assert provenance["input_hashes"] == expected_hashes, "Existing results do not match the current inputs."
        assert provenance["batch_size"] == EVAL_BATCH_SIZE
    else:
        assert provenance["protocol_sha256"] == raw_sha(PROTOCOL)
        protocol_inputs = provenance["protocol"]["inputs"]
        assert protocol_inputs["checkpoint"]["sha256"] == CHECKPOINT_SHA
        assert protocol_inputs["calibration"]["canonical_sha256"] == CALIBRATION_SHA
        assert protocol_inputs["manifest"]["canonical_sha256"] == EXPECTED_SPLIT_SHA
        assert provenance["protocol"]["settings"]["batch_size"] == EVAL_BATCH_SIZE
    completed = provenance["completed"]
    assert len(completed) == 10 and {row["configuration"] for row in completed} == set(CONFIGURATIONS)
    required = {f"{label}.{ext}" for label in CONFIGURATIONS for ext in ("json", "csv")}
    required |= {"paired_analysis.json", "run_provenance.json"}
    assert all((OFFICIAL_DIR / name).is_file() for name in required)
    sys.path.insert(0, str(REPO_DIR))
    from sparse_pet.paired_analysis import read_run
    predictions, raw_fingerprints = {}, {}
    for label in CONFIGURATIONS:
        predictions[label], fingerprints = read_run(OFFICIAL_DIR, label, "test")
        raw_fingerprints.update(fingerprints)
        result = json.loads((OFFICIAL_DIR / f"{label}.json").read_text())
        assert result["scores"]["n"] == 3669 and result["device"] == "cuda"
        assert result["split"] == "test"
        assert result["layer_indices"] == (None if label == "dense" else LAYERS)
        if label in ("dense", "unmasked"):
            assert result["mask_config"] is None
        else:
            selected = calibration["selected"][label]
            assert result["mask_config"] == {"method":"top_p" if label[0] in "AB" else "hybrid",
                    "p":selected["p"], "top_k":selected["top_k"],
                    "similarity_threshold":selected["similarity_threshold"], "grid_size":14, "tile_size":2}
        RESULTS[label] = result
    for entry in completed:
        label = entry["configuration"]
        assert entry["output_sha256"] == {f"{label}.{ext}":raw_fingerprints[f"{label}.{ext}"] for ext in ("json", "csv")}
    assert provenance["paired_analysis_sha256"] == raw_sha(OFFICIAL_DIR / "paired_analysis.json")
    dense_predictions = predictions["dense"]
    for label, values in predictions.items():
        assert set(values) == set(dense_predictions)
        assert all(values[k][0] == dense_predictions[k][0] for k in values)
    PAIRED = json.loads((OFFICIAL_DIR / "paired_analysis.json").read_text())
    assert PAIRED["split"] == "test" and PAIRED["n"] == 3669 and PAIRED["reference"] == "dense"
    assert set(PAIRED["configurations"]) == set(CONFIGURATIONS)
    assert set(PAIRED["comparisons"]) == set(CONFIGURATIONS) - {"dense"}
    for name, fingerprint in raw_fingerprints.items():
        assert PAIRED["input_sha256"][name] == fingerprint
    disagreements = sum(predictions["unmasked"][k][1] != dense_predictions[k][1] for k in dense_predictions)
    print("All 22 raw files present and all ten runs contain 3,669 test IDs; Dense/Unmasked disagreements:", disagreements)
    if disagreements:
        raise RuntimeError("Official Dense/Unmasked controls disagree. Investigate before interpreting results; do not tune parameters on test data.")
    OUTPUTS_OK = True


## Block 19 — Produce the final results table

Present accuracy and realized sparsity as percentages; present accuracy differences and confidence interval endpoints in **percentage points (pp)**. Macro-F1 remains on a 0–1 scale.

Unavailable diagnostic fields are left missing rather than invented. The official launcher does not produce latency measurements, so do not copy unrelated timing data into the test-results table.


In [ ]:
if OUTPUTS_OK:
    DERIVED_DIR.mkdir(parents=True, exist_ok=True)
    summary = []
    for label, result in RESULTS.items():
        scores, diagnostics = result["scores"], result["diagnostics"]
        sparse = label not in ("dense", "unmasked")
        achieved = 100 * diagnostics.get("realized_sparsity", 0.0)
        target = int(label.rsplit("_", 1)[1]) if sparse else 0
        comparison = PAIRED["comparisons"].get(label)
        summary.append({"configuration":label, "n_test":scores["n"], "accuracy_pct":100*scores["accuracy"],
                        "macro_f1":scores["macro_f1"], "loss":scores["loss"], "sparsity_pct":achieved,
                        "target_sparsity_pct":target, "within_3pp":abs(achieved-target)<=3 if sparse else True,
                        "retained_mass":diagnostics.get("retained_mass"),
                        "relative_output_error":diagnostics.get("relative_output_error"),
                        "delta_accuracy_pp":comparison["accuracy_delta_pp"] if comparison else 0.0,
                        "ci95_low_pp":comparison["ci95_delta_pp"][0] if comparison else None,
                        "ci95_high_pp":comparison["ci95_delta_pp"][1] if comparison else None,
                        "prediction_disagreements":comparison["prediction_disagreements"] if comparison else 0})
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(DERIVED_DIR / "official_test_summary.csv", index=False)
    display(summary_df)
    for limitation in PAIRED["limitations"]:
        print("Statistical note:", limitation)
    print("The current launcher does not measure latency. Do not add prior validation or CPU timings to the test results table.")
else:
    print("No verified official test results are available.")


## Block 20 — Generate official-test figures

Produce two figures: (1) test accuracy versus realized sparsity and (2) paired accuracy differences from Dense with marginal 95% bootstrap intervals. Export each figure as PNG and vector PDF.

These intervals are conditional on a single trained checkpoint, do not measure training-seed uncertainty, and are not corrected for multiple comparisons.


In [ ]:
if OUTPUTS_OK:
    plt.rcParams.update({"figure.dpi":140, "font.size":10, "axes.spines.top":False, "axes.spines.right":False})
    fig, ax = plt.subplots(figsize=(7.8, 4.8))
    dense_accuracy = float(summary_df.loc[summary_df.configuration == "dense", "accuracy_pct"].iloc[0])
    ax.axhline(dense_accuracy, color="#64748b", linestyle="--", label="Dense reference")
    colors = {"A":"#2563eb", "B":"#059669", "C":"#d97706", "D":"#9333ea"}
    for variant, color in colors.items():
        subset = summary_df[summary_df.configuration.str.startswith(variant + "_")].sort_values("sparsity_pct")
        ax.plot(subset.sparsity_pct, subset.accuracy_pct, marker="o", color=color, label=variant)
    ax.set(xlabel="Realized sparsity in modified layers (%)", ylabel="Top-1 test accuracy (%)",
           title="Official test: accuracy–sparsity comparison")
    ax.grid(alpha=0.16)
    ax.legend(ncol=5, fontsize=9)
    fig.tight_layout()
    for suffix in ["png", "pdf"]:
        fig.savefig(DERIVED_DIR / f"test_accuracy_vs_sparsity.{suffix}", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    interval_df = summary_df[summary_df.configuration != "dense"].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    y = np.arange(len(interval_df))
    ax.hlines(y, interval_df.ci95_low_pp, interval_df.ci95_high_pp, color="#2563eb", linewidth=2)
    ax.scatter(interval_df.delta_accuracy_pp, y, color="#0f172a", s=32, zorder=3)
    ax.axvline(0, color="#64748b", linestyle="--")
    ax.set_yticks(y, interval_df.configuration)
    ax.invert_yaxis()
    ax.set(xlabel="Accuracy difference from Dense (percentage points)",
           title="Official test: marginal paired bootstrap 95% intervals")
    ax.grid(axis="x", alpha=0.16)
    fig.tight_layout()
    for suffix in ["png", "pdf"]:
        fig.savefig(DERIVED_DIR / f"test_accuracy_delta_ci.{suffix}", bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("No official test data; no placeholder plots will be generated.")


## Block 21 — Archive the reproducible evaluation outputs

Package the unchanged 22 raw official-test files, derived summaries, figures, session logs, environment information, and configuration fingerprints into a Drive archive. Do not include model weights, the dataset, or access tokens.

Saving to Google Drive is not the same as pushing the notebook or code to GitHub. Commit those through the normal reviewed Git workflow.


In [ ]:
if OUTPUTS_OK:
    archive_parent = PROJECT_DRIVE / "deliveries"
    archive_parent.mkdir(parents=True, exist_ok=True)
    archive_path = archive_parent / f"{OFFICIAL_RUN_NAME}_{SESSION_TAG}.zip"
    entries = []
    for directory, prefix in [(OFFICIAL_DIR,"raw"), (DERIVED_DIR,"derived"), (SESSION_DIR,"session")]:
        entries.extend((path, str(Path(prefix) / path.relative_to(directory))) for path in directory.rglob("*") if path.is_file())
    entries += [(CALIBRATION, "config/calibration.json"), (MANIFEST, "config/split_manifest.json")]
    if ROUTE == "retrain":
        entries += [(PROTOCOL, "config/reviewed_protocol.json"),
                    (VALIDATION_DIR / "validation_provenance.json", "config/validation_provenance.json")]
    checksums = {relative:raw_sha(path) for path,relative in entries}
    with zipfile.ZipFile(archive_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for path, relative in entries:
            archive.write(path, relative)
        archive.writestr("sha256_manifest.json", json.dumps(checksums, indent=2))
    with zipfile.ZipFile(archive_path) as archive:
        assert archive.testzip() is None
    print("Complete archive saved:", archive_path)
    print("Archive SHA-256:", raw_sha(archive_path))
else:
    print("Official evaluation is not complete. Preparation/failure logs are saved in", SESSION_DIR)


## Troubleshooting

| Issue | Recommended action |
|---|---|
| Cannot find `best.pt` | Obtain the original `Sparge_Project_Backup.zip`; use retraining only if the original model truly cannot be recovered |
| Git clone fails | Verify repository access and authentication; do not use the PR URL as a clone URL |
| PR or full SHA changed | Reconfirm the approved commit and do not pull during an official run |
| Checkpoint or calibration hash mismatch | Do not substitute weights within V4; a new checkpoint requires separate calibration and approval |
| New reviewed-test runner missing | Commit the companion runner and tests to a feature branch, then pin that revision |
| Working tree is dirty | Review or commit local changes; avoid force-cleaning or overwriting files |
| Results directory already exists | Keep the existing directory and select a new experiment name after diagnosing any failure |
| New calibration cannot achieve a target budget | Expand the search based on **validation only**, or report the target as infeasible |
| Colab disconnects | Completed Drive writes remain; the original official launcher has no automatic resume |
| Want evidence of training speedup | This notebook does not supply it; a trainable sparse kernel and a controlled training benchmark would be required |

## Source references

- Uploaded repository: `RUN_NEXT_COLAB.md` (checkpoint backup instructions) and `README.md` (V4 status).
- `docs/OFFICIAL_TEST_WORKFLOW.md`, `docs/EXPERIMENT_FREEZE.md`, and `scripts/run_official_test.py` (original test procedure).
- `sparse_pet/train.py`, `calibrate.py`, `evaluate.py`, and `paired_analysis.py` (implementation invoked here).
- [GitHub: checking out pull requests locally](https://docs.github.com/en/pull-requests/how-tos/review-pull-requests/checking-out-pull-requests-locally)
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)

The notebook has been inspected offline, but real data downloads, GPU inference, model training, and final team approval must be completed in your runtime.
